# Unidad 2: Aprendizaje Automático 
## ✂️ Poda de Árboles de Decisión
### Inteligencia Artificial - Lic. en Sistemas - FCAD/UNER

![Diabetes](https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/notebooks/ml/images/pexels-teona-swift-6913823.jpg)

[Foto de Teona Swift](https://www.pexels.com/es-es/foto/ligero-persona-planta-mesa-6913823/)

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/ia-ls-fcad-uner/blob/main/notebooks/ml/07_DecisionTree_Pruned.ipynb)

## 🎯 ¿Qué vamos a aprender?

En el notebook anterior entrenamos un Árbol de Decisión **sin restricciones**. Vimos que puede memorizar los datos de entrenamiento, lo que se conoce como **overfitting**.

En este notebook vamos a aprender a **podar** el árbol: limitar su crecimiento para que generalice mejor.

Al finalizar, vas a poder:
- ✅ Entender qué es el overfitting y por qué es un problema
- ✅ Usar `max_depth` para limitar la profundidad del árbol
- ✅ Comparar un árbol sin podar vs uno podado
- ✅ Elegir la profundidad óptima con un gráfico de validación

---

## 🌳 ¿Qué es la Poda (Pruning)?

Un árbol sin restricciones crece hasta que **cada hoja contiene una sola muestra** (o las muestras son perfectamente separables). Esto hace que memorice el ruido del entrenamiento.

La **poda** consiste en detener el crecimiento del árbol antes de que se sobreajuste, ya sea:

| Técnica | Parámetro en scikit-learn | Descripción |
|---------|--------------------------|-------------|
| Limitar profundidad | `max_depth` | Máximo de niveles del árbol |
| Mínimo de muestras por nodo | `min_samples_split` | Mínimo de muestras para dividir un nodo |
| Mínimo de muestras por hoja | `min_samples_leaf` | Mínimo de muestras en cada hoja |
| Costo-complejidad | `ccp_alpha` | Penalización por complejidad (poda post-entrenamiento) |

> 📌 **Referencia:** [DataCamp - Decision Tree Classification in Python](https://www.datacamp.com/community/tutorials/decision-tree-classification-python)

## 📦 Paso 1: Importar Librerías

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import ConfusionMatrixDisplay

print('✅ Librerías importadas correctamente!')

## 📂 Paso 2: Cargar Datos y Preparar el Dataset

Usamos el mismo dataset de diabetes Pima Indians que en el notebook anterior.

In [ ]:
# Nombres de las columnas
col_names = ['pregnant', 'glucose', 'bp', 'skin',
             'insulin', 'bmi', 'pedigree', 'age', 'label']

# 📥 Cargar el dataset
pima = pd.read_csv('https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/data/ml/pima-indians-diabetes.csv', header=0, names=col_names)

print(f'📐 Dataset cargado: {pima.shape[0]} pacientes, {pima.shape[1]} columnas')
pima.head()

In [ ]:
# 🎯 Features y Target
feature_cols = ['pregnant', 'insulin', 'bmi', 'age', 'glucose', 'bp', 'pedigree']
X = pima[feature_cols]
y = pima['label']

# ✂️ División 70/30
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=111
)

print(f'🏋️  Entrenamiento: {X_train.shape[0]} muestras')
print(f'🧪  Prueba:        {X_test.shape[0]} muestras')

## ⚔️ Paso 3: Árbol Sin Podar vs Árbol Podado

Vamos a entrenar **dos modelos** y comparar su comportamiento:

| Modelo | Configuración | Riesgo |
|--------|--------------|--------|
| 🌳 **Sin podar** | `max_depth=None` (default) | ⚠️ Overfitting |
| ✂️ **Podado** | `max_depth=3` | ✅ Mejor generalización |

Con `max_depth=3`, el árbol puede hacer como máximo **3 preguntas** en cadena antes de decidir una clase.

In [ ]:
# 🌳 Árbol SIN podar
model_full = DecisionTreeClassifier(random_state=111)
model_full.fit(X_train, y_train)

# ✂️ Árbol PODADO con max_depth=3
model_pruned = DecisionTreeClassifier(max_depth=3, random_state=111)
model_pruned.fit(X_train, y_train)

print('✅ Ambos modelos entrenados!')
print(f'\n🌳 Árbol completo  — Profundidad: {model_full.get_depth():2d} | Hojas: {model_full.get_n_leaves():3d}')
print(f'✂️  Árbol podado    — Profundidad: {model_pruned.get_depth():2d} | Hojas: {model_pruned.get_n_leaves():3d}')

## 📊 Paso 4: Comparar Métricas

Ahora comparamos el rendimiento de ambos modelos en **entrenamiento** y **prueba**.

> 🔑 **Clave:** Un buen modelo tiene accuracy similar en entrenamiento y en prueba.  
> Si la accuracy de entrenamiento es mucho mayor → **overfitting**.

In [ ]:
def evaluar_modelo(nombre, modelo, X_tr, y_tr, X_te, y_te):
    """Imprime las métricas de un modelo en train y test."""
    y_pred = modelo.predict(X_te)
    acc_train = modelo.score(X_tr, y_tr)
    acc_test  = modelo.score(X_te, y_te)
    prec  = metrics.precision_score(y_te, y_pred)
    rec   = metrics.recall_score(y_te, y_pred)
    f1    = metrics.f1_score(y_te, y_pred)

    print(f'\n  {nombre}')
    print(f'    🏋️  Accuracy Entrenamiento: {acc_train:.4f}')
    print(f'    🧪  Accuracy Prueba:        {acc_test:.4f}  ← diferencia: {(acc_train-acc_test)*100:.1f}pp')
    print(f'    🔬  Precision: {prec:.4f} | 📡 Recall: {rec:.4f} | ⚖️  F1: {f1:.4f}')
    return y_pred

print('=' * 60)
print('           📊 COMPARACIÓN DE MODELOS')
print('=' * 60)
y_pred_full   = evaluar_modelo('🌳 Árbol Sin Podar', model_full,   X_train, y_train, X_test, y_test)
y_pred_pruned = evaluar_modelo('✂️  Árbol Podado (max_depth=3)', model_pruned, X_train, y_train, X_test, y_test)
print('=' * 60)

In [ ]:
# 📊 Visualización comparativa de métricas
metric_names = ['Accuracy\n(test)', 'Precision', 'Recall', 'F1']

def get_metrics(model, X_te, y_te):
    y_p = model.predict(X_te)
    return [
        metrics.accuracy_score(y_te, y_p),
        metrics.precision_score(y_te, y_p),
        metrics.recall_score(y_te, y_p),
        metrics.f1_score(y_te, y_p)
    ]

vals_full   = get_metrics(model_full,   X_test, y_test)
vals_pruned = get_metrics(model_pruned, X_test, y_test)

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, vals_full,   width, label='Sin podar', color='steelblue',  edgecolor='black')
bars2 = ax.bar(x + width/2, vals_pruned, width, label='Podado (depth=3)', color='mediumseagreen', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(metric_names, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Valor', fontsize=12)
ax.set_title('Comparación de Métricas: Árbol Completo vs Podado', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 🧩 Paso 5: Matrices de Confusión Comparadas

Visualizamos las matrices de confusión de ambos modelos lado a lado para entender qué tipo de errores comete cada uno.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_full,
    display_labels=['No Diabetes', 'Diabetes'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Árbol Sin Podar', fontsize=13)

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_pruned,
    display_labels=['No Diabetes', 'Diabetes'],
    cmap='Greens', ax=axes[1]
)
axes[1].set_title('Árbol Podado (max_depth=3)', fontsize=13)

plt.suptitle('Matrices de Confusión Comparadas', fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

## 🌲 Paso 6: Visualizar Ambos Árboles

Aquí podemos ver la diferencia visual entre un árbol profundo (complejo) y uno podado (simple e interpretable).

- El árbol sin podar es **enorme** y difícil de interpretar
- El árbol podado con `max_depth=3` es **legible** y explica su razonamiento

In [ ]:
# 🌳 Árbol sin podar (mostramos solo los primeros 3 niveles)
fig, ax = plt.subplots(figsize=(20, 7))
plot_tree(
    model_full,
    feature_names=feature_cols,
    class_names=['No Diabetes', 'Diabetes'],
    filled=True, rounded=True,
    max_depth=3, fontsize=9, ax=ax
)
ax.set_title(f'Árbol Sin Podar (profundidad real={model_full.get_depth()}) — mostrando primeros 3 niveles',
             fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ✂️ Árbol podado completo (max_depth=3 → es pequeño y legible)
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    model_pruned,
    feature_names=feature_cols,
    class_names=['No Diabetes', 'Diabetes'],
    filled=True, rounded=True,
    fontsize=11, ax=ax
)
ax.set_title('Árbol Podado completo (max_depth=3)', fontsize=10)
plt.tight_layout()
plt.show()

## 🔎 Lectura del Árbol Podado

Cada nodo del árbol muestra:

```
glucose <= 127.5         ← condición de división
gini = 0.459             ← impureza de Gini (0 = puro, 0.5 = máximo desorden)
samples = 537            ← cantidad de muestras en este nodo
value = [348, 189]       ← [No Diabetes, Diabetes]
class = No Diabetes      ← clase mayoritaria
```

- Rama **izquierda** → condición **verdadera** (glucose ≤ 127.5)
- Rama **derecha** → condición **falsa** (glucose > 127.5)
- **Color azul** → mayoría No Diabetes
- **Color naranja** → mayoría Diabetes
- **Intensidad** → qué tan dominante es la clase mayoritaria

## 🔬 Paso 7: Encontrar la Profundidad Óptima

¿Cómo elegimos el mejor valor de `max_depth`? Probamos varios valores y graficamos cómo cambia la accuracy en entrenamiento y prueba.

Buscamos el punto donde la accuracy de **prueba es máxima** y la brecha con entrenamiento es pequeña.

In [ ]:
depths = range(1, 15)
train_scores, test_scores = [], []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=111)
    m.fit(X_train, y_train)
    train_scores.append(m.score(X_train, y_train))
    test_scores.append(m.score(X_test, y_test))

best_depth = list(depths)[test_scores.index(max(test_scores))]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, train_scores, 'o-', label='Entrenamiento', color='steelblue', linewidth=2)
ax.plot(depths, test_scores,  's-', label='Prueba',         color='tomato',    linewidth=2)
ax.axvline(x=best_depth, color='green', linestyle='--', linewidth=2, label=f'Mejor depth={best_depth}')
ax.axvline(x=3, color='purple', linestyle=':', linewidth=2, label='depth=3 (modelo original)')

ax.fill_between(depths, train_scores, test_scores, alpha=0.1, color='red', label='Zona de overfitting')

ax.set_xlabel('Profundidad máxima (max_depth)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy vs Profundidad — Bias-Variance Tradeoff', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xticks(list(depths))
plt.tight_layout()
plt.show()

print(f'\n🏆 Profundidad óptima: max_depth={best_depth} → Accuracy test={max(test_scores):.4f}')
print(f'📌 Modelo original: max_depth=3           → Accuracy test={test_scores[2]:.4f}')

## 🎓 Resumen y Conclusiones

### ¿Qué aprendimos?

| Concepto | Árbol Sin Podar | Árbol Podado |
|----------|----------------|-------------|
| Profundidad | Sin límite 📈 | Controlada ✂️ |
| Accuracy Train | ~100% 🚨 | Menor pero razonable ✅ |
| Accuracy Test | Menor | Igual o mejor ✅ |
| Interpretabilidad | Muy difícil 😵 | Fácil de leer 📖 |
| Overfitting | ⚠️ Probable | ✅ Controlado |

### 🔑 Regla práctica:
> Un árbol con **`max_depth` entre 3 y 5** suele ser un buen punto de partida.  
> Usá el gráfico de **Accuracy vs Profundidad** para elegir el valor óptimo para tu dataset.

---

### 🚀 ¿Qué sigue?

- 🖼️ **Visualizar el árbol con Graphviz**: exportar el árbol como imagen de alta calidad
- 🌲🌲🌲 **Random Forests**: combinar cientos de árboles podados para mayor robustez
- 📊 **Validación cruzada**: evaluar el modelo en múltiples particiones del dataset

---
> 📚 **Referencia:** [scikit-learn - Tree Pruning](https://scikit-learn.org/stable/modules/tree.html#minimal-cost-complexity-pruning)

---

© 2026 Cátedra Inteligencia Artificial — Lic. en Sistemas

Este notebook se distribuye bajo licencia [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).